In [13]:
import pandas as pd
import numpy as np
import sqlite3
import os

# Đường dẫn tới thư mục data/raw
raw_path = '../data/raw/archive (2)'

print("⏳ Đang nạp các bảng dữ liệu...")
df_orders = pd.read_csv(f"{raw_path}/olist_orders_dataset.csv")
df_items = pd.read_csv(f"{raw_path}/olist_order_items_dataset.csv")
df_payments = pd.read_csv(f"{raw_path}/olist_order_payments_dataset.csv")
df_products = pd.read_csv(f"{raw_path}/olist_products_dataset.csv")
df_customers = pd.read_csv(f"{raw_path}/olist_customers_dataset.csv")
df_reviews = pd.read_csv(f"{raw_path}/olist_order_reviews_dataset.csv")
df_translation = pd.read_csv(f"{raw_path}/product_category_name_translation.csv")

print("✅ Đã nạp thành công 7 bảng quan trọng nhất!")

⏳ Đang nạp các bảng dữ liệu...
✅ Đã nạp thành công 7 bảng quan trọng nhất!


In [14]:
# 1.1 Kiểm tra kích thước ban đầu của từng bảng
tables = {
    'orders': df_orders, 'items': df_items, 'payments': df_payments,
    'products': df_products, 'customers': df_customers, 'reviews': df_reviews
}
for name, df in tables.items():
    print(f"Bảng {name:10}: {df.shape[0]:,} dòng | {df.shape[1]} cột")

# 1.2 Ép kiểu ngày tháng cho bảng orders
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df_orders[col] = pd.to_datetime(df_orders[col])

print("\n✅ Đã ép kiểu datetime thành công cho 5 cột thời gian trong bảng orders!")


Bảng orders    : 99,441 dòng | 8 cột
Bảng items     : 112,650 dòng | 7 cột
Bảng payments  : 103,886 dòng | 5 cột
Bảng products  : 32,951 dòng | 9 cột
Bảng customers : 99,441 dòng | 5 cột
Bảng reviews   : 99,224 dòng | 7 cột

✅ Đã ép kiểu datetime thành công cho 5 cột thời gian trong bảng orders!


In [15]:
# 2.1 Kiểm tra trùng lặp khóa chính (Primary Key)
print(f"Trùng lặp order_id trong orders: {df_orders['order_id'].duplicated().sum()}")
print(f"Trùng lặp customer_id trong customers: {df_customers['customer_id'].duplicated().sum()}")
print(f"Trùng lặp product_id trong products: {df_products['product_id'].duplicated().sum()}")

# 2.2 Kiểm tra dữ liệu 'mồ côi' (Orphan records)
# Kiểm tra xem có order_id nào trong order_items mà KHÔNG tồn tại trong orders không
valid_order_ids = set(df_orders['order_id'])
orphan_items = (~df_items['order_id'].isin(valid_order_ids)).sum()
orphan_payments = (~df_payments['order_id'].isin(valid_order_ids)).sum()

print(f"Số dòng items mồ côi (không có order gốc): {orphan_items}")
print(f"Số dòng payments mồ côi: {orphan_payments}")

Trùng lặp order_id trong orders: 0
Trùng lặp customer_id trong customers: 0
Trùng lặp product_id trong products: 0
Số dòng items mồ côi (không có order gốc): 0
Số dòng payments mồ côi: 0


In [16]:
# 3.1 Nối bảng products với bảng translation để lấy tên tiếng Anh
df_products_clean = df_products.merge(df_translation, on='product_category_name', how='left')

# Nếu sản phẩm nào thiếu bản dịch, điền là 'others'
df_products_clean['product_category_name_english'] = df_products_clean['product_category_name_english'].fillna('others')

# 3.2 Lược bỏ các cột thông số quá chi tiết không dùng cho Dashboard/AI để làm nhẹ dữ liệu
cols_to_drop = [
    'product_name_lenght', 
    'product_description_lenght', 
    'product_photos_qty'
]
df_products_clean = df_products_clean.drop(columns=cols_to_drop)

print(f"✅ Bảng products sau khi dịch và tinh gọn: {df_products_clean.shape}")
df_products_clean[['product_id', 'product_category_name_english']].head(3)

✅ Bảng products sau khi dịch và tinh gọn: (32951, 7)


,product_id,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure


In [17]:
# 4.1 Kiểm tra logic chuỗi thời gian của bảng orders
# Quy tắc nghiệp vụ: Ngày nhận hàng thực tế PHẢI diễn ra SAU ngày đặt hàng
invalid_dates = df_orders[
    df_orders['order_delivered_customer_date'] < df_orders['order_purchase_timestamp']
]
print(f"Số đơn có ngày giao trước ngày đặt (lỗi logic): {len(invalid_dates)}")

# Nếu có đơn lỗi logic thời gian, loại bỏ khỏi phân tích
if len(invalid_dates) > 0:
    df_orders = df_orders.drop(invalid_dates.index)

# 4.2 Xử lý review_comment_title / review_comment_message bị null
# Đánh giá không viết chữ là bình thường trong TMĐT -> Điền chuỗi rỗng
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('')

print("✅ Đã làm sạch logic thời gian và xử lý giá trị khuyết thiếu!")

Số đơn có ngày giao trước ngày đặt (lỗi logic): 0
✅ Đã làm sạch logic thời gian và xử lý giá trị khuyết thiếu!


In [18]:
# Kiểm tra phân phối giá trị của sản phẩm (price) và cước phí (freight_value)
price_stats = df_items[['price', 'freight_value']].describe()
print("📊 Thống kê mô tả price và freight_value:")
display(price_stats)

# Tính ngưỡng IQR cho cột price
Q1 = df_items['price'].quantile(0.25)
Q3 = df_items['price'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outlier_count = (df_items['price'] > upper_bound).sum()
print(f"\nSố lượng đơn có giá cao hơn ngưỡng trên ({upper_bound:.2f}): {outlier_count} đơn")

# 💡 Quyết định nghiệp vụ (Business Context):
# Giá cao trong TMĐT là đơn hàng đắt tiền/mua sỉ hợp lệ (không phải lỗi nhập liệu) -> GIỮ NGUYÊN, không xóa.
# Chỉ lọc điều kiện: Giá tiền và cước ship phải lớn hơn 0
df_items = df_items[(df_items['price'] > 0) & (df_items['freight_value'] >= 0)]
print("✅ Đã xác thực Outlier theo bối cảnh kinh doanh!")

📊 Thống kê mô tả price và freight_value:


,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000



Số lượng đơn có giá cao hơn ngưỡng trên (277.40): 8427 đơn
✅ Đã xác thực Outlier theo bối cảnh kinh doanh!


In [19]:
# Đường dẫn file SQLite Database
db_path = '../data/ecommerce.db'
conn = sqlite3.connect(db_path)

print("💾 Đang ghi dữ liệu chuẩn hóa vào SQLite Database...")

df_orders.to_sql('orders', conn, if_exists='replace', index=False)
df_items.to_sql('order_items', conn, if_exists='replace', index=False)
df_payments.to_sql('order_payments', conn, if_exists='replace', index=False)
df_products_clean.to_sql('products', conn, if_exists='replace', index=False)
df_customers.to_sql('customers', conn, if_exists='replace', index=False)
df_reviews.to_sql('order_reviews', conn, if_exists='replace', index=False)

conn.close()

# Kiểm tra lại kích thước file db vừa tạo
db_size_mb = os.path.getsize(db_path) / (1024 * 1024)
print(f"🎉 HOÀN THÀNH TOÀN BỘ QUY TRÌNH!")
print(f"📁 Database sạch đã sẵn sàng tại: {db_path} (Dung lượng: ~{db_size_mb:.2f} MB)")

💾 Đang ghi dữ liệu chuẩn hóa vào SQLite Database...
🎉 HOÀN THÀNH TOÀN BỘ QUY TRÌNH!
📁 Database sạch đã sẵn sàng tại: ../data/ecommerce.db (Dung lượng: ~65.23 MB)
